# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShivanshRastogi315/FlyRank_Internship-repo/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

We construct the vector by separating features from the target, imputing missing values (zeros for missing metrics, penalty ranks for missing positions), and applying One-Hot Encoding to categorical fields so mathematical algorithms can process them.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Target Definition (Binary classification for anomaly detection)
df['is_ai_traffic'] = (df['ai_sessions_90d'] > 0).astype(int)

# Fill Missing Values
df['search_volume'] = df['search_volume'].fillna(0)
df['avg_position'] = df['avg_position'].fillna(100) # 100 acts as a penalty for unranked pages

# One-Hot Encoding for Categoricals
categorical_cols = ['content_type', 'main_intent']
X_categorical = pd.get_dummies(df[categorical_cols], drop_first=True)

# Numerical Features
numerical_cols = ['search_volume', 'word_count', 'content_age_days', 'days_since_last_update']
X_numerical = df[numerical_cols]

# Combine into Final Feature Vector (X)
X = pd.concat([X_numerical, X_categorical], axis=1)
y = df['is_ai_traffic']

print(f"Feature Vector Shape: {X.shape}")

Feature Vector Shape: (30000, 9)


## 2. Feature notes (meaning, missing, categorical, available-when?)

search_volume: Broad market demand. Missing values filled with 0. Exists prior to the 90-day prediction window.

avg_position: Current SERP rank. Missing values filled with 100. Exists prior to the prediction window.

content_type / main_intent: Structural context flags. One-hot encoded. Strictly known at the exact moment of publishing.

In [3]:
# Verify no nulls exist in the final vector
missing_in_X = X.isnull().sum().sum()
print(f"Total missing values in Feature Vector: {missing_in_X}")
print("Features ready for ML processing:\n", X.columns.tolist())

Total missing values in Feature Vector: 7699
Features ready for ML processing:
 ['search_volume', 'word_count', 'content_age_days', 'days_since_last_update', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional']


## 3. The leakage hunt

Data leakage occurs when a feature accidentally contains information from the future or the target itself. We hunt for this by checking direct linear correlations. If any feature correlates suspiciously high with is_ai_traffic (e.g., > 0.6), it must be investigated for leakage.

In [4]:
# Merge X and y temporarily to check correlations
leakage_check = X.copy()
leakage_check['TARGET'] = y

correlations = leakage_check.corr()['TARGET'].sort_values(ascending=False)
print("--- LEAKAGE HUNT: CORRELATION WITH TARGET ---")
print(correlations.drop('TARGET')) # Exclude self-correlation

--- LEAKAGE HUNT: CORRELATION WITH TARGET ---
word_count                      0.274387
days_since_last_update          0.137362
content_type_keyword article    0.027438
main_intent_navigational        0.003613
main_intent_informational      -0.001864
main_intent_transactional      -0.004086
content_type_feedly article    -0.010575
search_volume                  -0.016341
content_age_days               -0.024388
Name: TARGET, dtype: float64


## 4. What I excluded and why

ai_sessions_90d: Excluded because it is the target label.

sessions_90d / clicks_90d / impressions_90d: Excluded because they are parallel outcomes occurring simultaneously with the target. Including them allows the model to "cheat" by inferring that high overall traffic guarantees high AI traffic.

trend_direction: Excluded as it is calculated post-facto.

In [5]:
# Explicitly list excluded columns to enforce the contract
excluded_fields = ['ai_sessions_90d', 'sessions_90d', 'clicks_90d', 'impressions_90d', 'trend_direction', 'client_id', 'content_id']
accidental_inclusions = [col for col in excluded_fields if col in X.columns]

print(f"Excluded fields accidentally in Feature Vector: {len(accidental_inclusions)}")
if accidental_inclusions:
    print(f"WARNING: Leakage detected in columns: {accidental_inclusions}")
else:
    print("Feature Vector is clean and sealed.")

Excluded fields accidentally in Feature Vector: 0
Feature Vector is clean and sealed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.